In [3]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm

import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Upload the Dataset
This script creates a FiftyOne dataset from the local screenshots directory. It scans for all image files in the screenshots folder and its subdirectories.

In [4]:
# Set up paths
image_dir = Path('../../data/intermediate/v1/screenshots')

# Create a new dataset
dataset_name = "jaguar_detection"
if fo.dataset_exists(dataset_name):
    print(f"Loading existing dataset: {dataset_name}")
    dataset = fo.load_dataset(dataset_name)
    print(f"Dataset contains {len(dataset)} samples")
else:
    print(f"Creating new dataset: {dataset_name}")
    dataset = fo.Dataset(name=dataset_name, persistent=True)
    
    # Add all images from directory and subdirectories
    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        image_paths.extend(image_dir.rglob(ext))
    
    print(f"Found {len(image_paths)} images")
    
    samples = []
    for img_path in tqdm(image_paths, desc="Adding images"):
        sample = fo.Sample(filepath=str(img_path))
        samples.append(sample)
    
    dataset.add_samples(samples)
    print(f"Added {len(dataset)} samples to dataset")

Loading existing dataset: jaguar_detection
Dataset contains 2319 samples


# Run Grounding-Dino
This script loads a Grounding DINO zero-shot object detection model from the FiftyOne model zoo, configured to detect either "jaguar's whole body" or "Close-up of a jaguar's head" based on the DETECTION_TYPE flag.   
It runs the model on the dataset, saving predictions in the appropriate raw_bboxes_body or raw_bboxes_head field, using a confidence threshold of 0.2 and a text similarity threshold of 0.6.  
It then selects the best detection depending on the chosen detection type and removes the raw bounding box field after processing.

### Test data set


In [3]:
# Configuration
DETECTION_TYPE = "body"  # or "head" - set this flag to choose processing type
USE_TEST_DATASET = False  # Set to False to use full dataset

# Set device - use MPS for macOS if available, otherwise fallback to CPU
import torch
# if torch.backends.mps.is_available():
#     device = "mps"
#     print("Using Metal Performance Shaders (MPS) device")
# else:
device = "cpu"
print("Using CPU device")

# Select dataset
# working_dataset = test_dataset if USE_TEST_DATASET else dataset
working_dataset = dataset
print(f"Working with {'test' if USE_TEST_DATASET else 'full'} dataset ({len(working_dataset)} samples)")

# Load appropriate model based on detection type
model = foz.load_zoo_model(
    "zero-shot-detection-transformer-torch",
    name_or_path="IDEA-Research/grounding-dino-base",
    classes=["jaguar"],
    device=device
)

# Define the name of the bboxes field
raw_bboxes_name = f"raw_bboxes_{DETECTION_TYPE}"

# run model with batch_size=1 to avoid batching issues
working_dataset.apply_model(
    model,
    label_field=raw_bboxes_name,
    confidence_thresh=0.2,
    batch_size=1
)

print(f"✓ Model inference complete on {len(working_dataset)} samples")

Using Metal Performance Shaders (MPS) device
Working with full dataset (2319 samples)
 100% |███████████████| 2319/2319 [1.9h elapsed, 0s remaining, 0.3 samples/s]     
✓ Model inference complete on 2319 samples


In [4]:
# Helper functions for selecting best detection
def select_best_detection_body(dataset, raw_bboxes_field_name="raw_bboxes_body"):
    """Select detection with largest area as best body detection"""
    for sample in tqdm(dataset, desc="Selecting best body detection"):
        # Check if field exists and has detections
        if raw_bboxes_field_name not in sample or not sample[raw_bboxes_field_name]:
            continue
        
        raw_bboxes = sample[raw_bboxes_field_name]
        if raw_bboxes.detections:
            best_detection = max(
                raw_bboxes.detections,
                key=lambda d: d.bounding_box[2] * d.bounding_box[3],
            )
            sample["bboxes_body"] = fo.Detections(detections=[best_detection])
            sample.save()

def select_best_detection_head(dataset, raw_bboxes_field_name="raw_bboxes_head"):
    """Select detection with highest confidence as best head detection"""
    for sample in tqdm(dataset, desc="Selecting best head detection"):
        # Check if field exists and has detections
        if raw_bboxes_field_name not in sample or not sample[raw_bboxes_field_name]:
            continue
        
        raw_bboxes = sample[raw_bboxes_field_name]
        if raw_bboxes.detections:
            best_detection = max(
                raw_bboxes.detections,
                key=lambda d: d.confidence,
            )
            sample["bboxes_head"] = fo.Detections(detections=[best_detection])
            sample.save()

if DETECTION_TYPE == "body":
    # If you computed bboxes for whole body
    select_best_detection_body(working_dataset, raw_bboxes_field_name=raw_bboxes_name)
else:
    # If you computed bboxes for head only
    select_best_detection_head(working_dataset, raw_bboxes_field_name=raw_bboxes_name)

# Remove raw bboxes (only if field exists)
if raw_bboxes_name in working_dataset.get_field_schema():
    working_dataset.delete_sample_field(raw_bboxes_name)

print(f"✓ Best detection selection complete")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Selecting best body detection: 100%|██████████| 2319/2319 [00:06<00:00, 357.41it/s]



✓ Best detection selection complete


In [5]:
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"
samples_with_detections = working_dataset.exists(detection_field)
total_samples = len(working_dataset)
detected_samples = len(samples_with_detections)

print(f"Detection Results:")
print(f"  Total samples: {total_samples}")
print(f"  Samples with detections: {detected_samples}")
print(f"  Samples without detections: {total_samples - detected_samples}")
print(f"  Detection rate: {detected_samples/total_samples*100:.1f}%")

Detection Results:
  Total samples: 2319
  Samples with detections: 1367
  Samples without detections: 952
  Detection rate: 58.9%


In [6]:
import shutil

# Save positive and negative detections into separate subfolders with folder-based prefixes
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"

positive_dir = Path('../../data/intermediate/v1/screenshots/positive_detections')
negative_dir = Path('../../data/intermediate/v1/screenshots/negative_detections')

positive_dir.mkdir(parents=True, exist_ok=True)
negative_dir.mkdir(parents=True, exist_ok=True)

positive_count = 0
negative_count = 0

for sample in tqdm(working_dataset, desc="Organizing detections"):
    img_path = Path(sample.filepath)
    
    # Get the parent folder name to use as prefix
    folder_name = img_path.parent.name
    prefixed_filename = f"{folder_name}_{img_path.name}"
    
    # Check if sample has detections
    if sample[detection_field] and sample[detection_field].detections:
        # Positive detection
        dest = positive_dir / prefixed_filename
        positive_count += 1
    else:
        # Negative detection
        dest = negative_dir / prefixed_filename
        negative_count += 1
    
    # Copy file to appropriate folder
    shutil.copy(img_path, dest)

print(f"✓ Detections organized into folders")
print(f"  Total processed: {len(working_dataset)}")
print(f"  Positive: {positive_count} images -> {positive_dir}")
print(f"  Negative: {negative_count} images -> {negative_dir}")
print(f"  Positive folder: {len(list(positive_dir.glob('*')))} files")
print(f"  Negative folder: {len(list(negative_dir.glob('*')))} files")

Organizing detections: 100%|██████████| 2319/2319 [00:01<00:00, 1189.94it/s]

✓ Detections organized into folders
  Total processed: 2319
  Positive: 1367 images -> ../../data/intermediate/v1/screenshots/positive_detections
  Negative: 952 images -> ../../data/intermediate/v1/screenshots/negative_detections
  Positive folder: 1377 files
  Negative folder: 962 files


In [ ]:
# Define fields based on detection type
prompt_field = "bboxes_head" if DETECTION_TYPE == "head" else "bboxes_body"
label_field = "segmentations_head" if DETECTION_TYPE == "head" else "segmentations_body"

# Filter to only positive detections (images with jaguars)
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"
positive_dataset = working_dataset.exists(detection_field)

print(f"Filtering to positive detections: {len(positive_dataset)} images")

# Load the segmentation model
model = foz.load_zoo_model("segment-anything-vitb-torch", device=device)

# Apply the model only to positive detections
positive_dataset.apply_model(
    model,
    label_field=label_field,
    prompt_field=prompt_field
)

print(f"✓ Segmentation complete on {len(positive_dataset)} positive images")


Filtering to positive detections: 1367 images
 100% |███████████████| 1367/1367 [12.4m elapsed, 0s remaining, 2.3 samples/s]      
✓ Segmentation complete on 1367 positive images
Directory '../../data/intermediate/v1/fo_dataset' already exists; export will be merged with existing files
Exporting samples...
 100% |███████████████| 1367/1367 [12.4m elapsed, 0s remaining, 2.3 samples/s]      
✓ Segmentation complete on 1367 positive images
Directory '../../data/intermediate/v1/fo_dataset' already exists; export will be merged with existing files
Exporting samples...
 100% |██████████████████| 2319/2319 [318.7ms elapsed, 0s remaining, 7.3K docs/s]      
 100% |██████████████████| 2319/2319 [318.7ms elapsed, 0s remaining, 7.3K docs/s]      


In [18]:
storage_dir = Path('../../data/intermediate/v1/fo_jaguars')

post_segmentation_view = fo.load_dataset("jaguar_detection")

positive_field = "bboxes_body"
positive_samples_view = post_segmentation_view.exists(positive_field)

positive_samples_view.export(
    export_dir=str(storage_dir / "segmented_jaguars"),
    dataset_type=fo.types.FiftyOneDataset,
    export_media=True,
    rel_dir=str(image_dir)
)
print(f"✓ Exported {len(positive_samples_view)} positive samples to {storage_dir / 'segmented_jaguars'}")

session = fo.launch_app(positive_samples_view)

Exporting samples...
 100% |██████████████████| 1367/1367 [1.1s elapsed, 0s remaining, 1.2K docs/s]         
 100% |██████████████████| 1367/1367 [1.1s elapsed, 0s remaining, 1.2K docs/s]         
✓ Exported 1367 positive samples to ../../data/intermediate/v1/fo_jaguars/segmented_jaguars
✓ Exported 1367 positive samples to ../../data/intermediate/v1/fo_jaguars/segmented_jaguars


# Add Jaguar ID Labels
Match sample filenames to jaguar IDs from the cleaned labels CSV and add them as metadata to each sample.


## Step 1: Load Labels and Create Mapping
Load the cleaned labels CSV and create a mapping from filename to jaguar ID for verification.


In [7]:
import pandas as pd

labels_path = Path('../../data/intermediate/v1/cleaned_labels.csv')
print(f"Loading labels from {labels_path}...")
labels_df = pd.read_csv(labels_path)

# Convert FILE PATH to mapping keys: "sites/SITE 11/CAM B/DSCF0016.AVI" -> "SITE_11_CAM_B_DSCF0016"
def extract_filepath_key(file_path):
    file_path = str(file_path)
    # Remove "sites/" prefix
    if "sites/" in file_path:
        file_path = file_path.split("sites/", 1)[1]
    # Remove extension
    file_path = str(Path(file_path).with_suffix(''))
    # Replace path separators and spaces with underscores
    key = file_path.replace("/", "_").replace(" ", "_")
    return key

# Create mapping using extracted filepaths
file_to_jaguar = {}
for idx, row in labels_df.iterrows():
    filepath_key = extract_filepath_key(row['FILE PATH'])
    jaguar_id = row['JAGUAR ID']
    file_to_jaguar[filepath_key] = jaguar_id

print(f"Loaded {len(labels_df)} labels from CSV")
print(f"Found {len(file_to_jaguar)} file-to-jaguar mappings")

print("\nSample file-to-jaguar mappings:")
for i, (fpath, jid) in enumerate(list(file_to_jaguar.items())[:10]):
    print(f"  {fpath} -> {jid}")


Loading labels from ../../data/intermediate/v1/cleaned_labels.csv...
Loaded 271 labels from CSV
Found 252 file-to-jaguar mappings

Sample file-to-jaguar mappings:
  SITE_11_CAM_B_DSCF0016 -> U32
  SITE_4_PHOTOS_ID_DSCF0358 -> U17
  SITE_4_PHOTOS_ID_DSCF0357 -> U17
  SITE_8_CAM_A_DSCF0058 -> U7
  SITE_8_CAM_A_DSCF0188 -> BORORO
  SITE_8_CAM_A_DSCF0041 -> BUHEE
  SITE_8_CAM_A_DSCF0237 -> BUHEE
  SITE_8_CAM_A_DSCF0236 -> BUHEE
  SITE_8_CAM_A_DSCF0235 -> BUHEE
  SITE_8_CAM_A_DSCF0234 -> BUHEE


## Step 2: Load Dataset and Check Filename Matches
Load the positive samples dataset and verify that filenames can be matched to jaguar IDs.


In [ ]:
# Load the segmented_jaguars dataset from camera-trap-footage/data/intermediate/v1/fo_jaguars/segmented_jaguars
storage_dir = Path('../../data/intermediate/v1/fo_jaguars')

segmented_dataset = fo.Dataset.from_dir(
    dataset_dir=str(storage_dir / "segmented_jaguars"),
    dataset_type=fo.types.FiftyOneDataset,
    name="segmented_jaguars_loaded"
)

print(f"Loaded {len(segmented_dataset)} samples from segmented_jaguars dataset")
print(f"Fields: {segmented_dataset.get_field_schema().keys()}")



Importing samples...
 100% |███████████████| 1367/1367 [46.5ms elapsed, 0s remaining, 29.4K samples/s]   
 100% |███████████████| 1367/1367 [46.5ms elapsed, 0s remaining, 29.4K samples/s]   
Loaded 1367 samples from segmented_jaguars dataset
Fields: odict_keys(['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bboxes_body', 'segmentations_body'])
Loaded 1367 samples from segmented_jaguars dataset
Fields: odict_keys(['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bboxes_body', 'segmentations_body'])


## Step 3: Apply Labels and Visualize
Add the jaguar ID labels to all matched samples and launch FiftyOne for visualization.


In [ ]:
# Add jaguar ID labels to all matched samples
import pandas as pd

labeled_count = 0
unlabeled_count = 0
unlabeled_files = []

for sample in tqdm(segmented_dataset, desc="Adding jaguar ID labels"):
    # Extract the key from filepath: get parent folder name + filename without extension
    filepath = Path(sample.filepath)
    folder_name = filepath.parent.name
    filename_no_ext = filepath.stem
    
    # Remove any frame suffix (e.g., "_frame_001") to get the base video name
    # Pattern: SITE_X_CAM_Y_VIDEONAME_frame_NNN -> SITE_X_CAM_Y_VIDEONAME
    base_name = filename_no_ext.rsplit('_frame_', 1)[0] if '_frame_' in filename_no_ext else filename_no_ext
    
    # Try different key formats
    possible_keys = [
        base_name,  # Direct match
        folder_name,  # Folder name as key
        f"{folder_name}_{filepath.stem}",  # Folder + filename
    ]
    
    jaguar_id = None
    for key in possible_keys:
        if key in file_to_jaguar:
            jaguar_id = file_to_jaguar[key]
            break
    
    # Check if jaguar_id is valid (not None and not NaN)
    if jaguar_id is not None and not (isinstance(jaguar_id, float) and pd.isna(jaguar_id)):
        sample["jaguar_id"] = fo.Classification(label=str(jaguar_id))
        sample.save()
        labeled_count += 1
    else:
        unlabeled_count += 1
        if len(unlabeled_files) < 5:
            unlabeled_files.append(base_name)

print(f"\n✓ Labels added:")
print(f"  Labeled: {labeled_count} samples")
print(f"  Unlabeled: {unlabeled_count} samples")
print(f"  Coverage: {labeled_count/(labeled_count+unlabeled_count)*100:.1f}%")

if unlabeled_files:
    print(f"\nSample unlabeled files (first 5):")
    for f in unlabeled_files:
        print(f"  {f}")

# Launch FiftyOne for visualization
print("\nLaunching FiftyOne app...")
session = fo.launch_app(segmented_dataset)

Adding jaguar ID labels:   0%|          | 0/1367 [00:00<?, ?it/s]



ValidationError: StringField only accepts string values

## Export Labeled Dataset
Load the segmented jaguars dataset, add jaguar ID labels, and export as a new labeled dataset.


In [12]:
# Export the labeled dataset with media files
export_dir = Path('../../data/intermediate/v1/fo_jaguars/labeled_segmented_jaguars')
export_dir.parent.mkdir(parents=True, exist_ok=True)

segmented_dataset.export(
    export_dir=str(export_dir),
    dataset_type=fo.types.FiftyOneDataset,
    export_media=True
)

print(f"✓ Labeled dataset exported:")
print(f"  Location: {export_dir}")
print(f"  Total samples: {len(segmented_dataset)}")
print(f"  With segmentations and jaguar ID labels")

Exporting samples...
 100% |██████████████████| 1367/1367 [675.4ms elapsed, 0s remaining, 2.0K docs/s]      
✓ Labeled dataset exported:
  Location: ../../data/intermediate/v1/fo_jaguars/labeled_segmented_jaguars
  Total samples: 1367
  With segmentations and jaguar ID labels
 100% |██████████████████| 1367/1367 [675.4ms elapsed, 0s remaining, 2.0K docs/s]      
✓ Labeled dataset exported:
  Location: ../../data/intermediate/v1/fo_jaguars/labeled_segmented_jaguars
  Total samples: 1367
  With segmentations and jaguar ID labels


In [ ]:
# Load the labeled segmented dataset and launch FiftyOne session
labeled_dir = Path('../../data/intermediate/v1/fo_jaguars/labeled_segmented_jaguars')

labeled_dataset = fo.Dataset.from_dir(
    dataset_dir=str(labeled_dir),
    dataset_type=fo.types.FiftyOneDataset,
    name="labeled_segmented_jaguars_view"
)

print(f"Loaded {len(labeled_dataset)} samples from labeled dataset")
print(f"Fields: {list(labeled_dataset.get_field_schema().keys())}")

# Check for jaguar_id field
if "jaguar_id" in labeled_dataset.get_field_schema():
    labeled_samples = labeled_dataset.exists("jaguar_id")
    print(f"Samples with jaguar_id: {len(labeled_samples)}")
else:
    print("Warning: jaguar_id field not found in dataset")

# Launch FiftyOne app
session = fo.launch_app(labeled_dataset)

Importing samples...
 100% |███████████████| 1367/1367 [48.0ms elapsed, 0s remaining, 28.5K samples/s]   
 100% |███████████████| 1367/1367 [48.0ms elapsed, 0s remaining, 28.5K samples/s]   
Loaded 1367 samples from labeled dataset
Fields: ['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bboxes_body', 'segmentations_body', 'jaguar_id']
Samples with jaguar_id: 1120
Loaded 1367 samples from labeled dataset
Fields: ['id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bboxes_body', 'segmentations_body', 'jaguar_id']
Samples with jaguar_id: 1120
